<a href="https://colab.research.google.com/github/nitijain18/style-buddy/blob/main/model4_sb_pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets


In [2]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/nguyngiabol/dress-pattern-dataset")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: nitijain13
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nguyngiabol/dress-pattern-dataset


100%|██████████| 118M/118M [00:00<00:00, 124MB/s]


In [20]:
import os, shutil

old_path = "/content/dress-pattern-dataset/data_pattern"
new_path = "/content/pattern_merged"

mapping = {
    "animal": "animal",
    "cartoon": "cartoon",
    "floral": "floral",
    "geometry": "geometric",
    "squares": "geometric",
    "ikat": "ethnic",
    "tribal": "ethnic",
    "plain": "plain",
    "polka dot": "polka_dot",
    "stripes": "stripes"
}

os.makedirs(new_path, exist_ok=True)

for old_class, new_class in mapping.items():
    old_folder = os.path.join(old_path, old_class)
    new_folder = os.path.join(new_path, new_class)
    os.makedirs(new_folder, exist_ok=True)

    for file in os.listdir(old_folder):
        src = os.path.join(old_folder, file)
        dst = os.path.join(new_folder, old_class + "_" + file)
        shutil.copy(src, dst)

for folder in os.listdir(new_path):
    print(folder, len(os.listdir(os.path.join(new_path, folder))))

polka_dot 498
animal 351
geometric 778
stripes 500
floral 495
plain 497
cartoon 260
ethnic 851


In [21]:
import tensorflow as tf


train_ds = tf.keras.utils.image_dataset_from_directory(
     "/content/pattern_merged",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
     "/content/pattern_merged",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

Found 4230 files belonging to 8 classes.
Using 3384 files for training.
Found 4230 files belonging to 8 classes.
Using 846 files for validation.


In [22]:
class_names = train_ds.class_names
print(class_names)

['animal', 'cartoon', 'ethnic', 'floral', 'geometric', 'plain', 'polka_dot', 'stripes']


In [23]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [35]:
print(class_names)

['animal', 'cartoon', 'ethnic', 'floral', 'geometric', 'plain', 'polka_dot', 'stripes']


In [36]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from tensorflow.keras.applications.resnet50 import preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(8, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,312 (91.99 MB)

 Trainable params: 526,600 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_pattern_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

In [39]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 49s 349ms/step - accuracy: 0.3918 - loss: 1.7326 - val_accuracy: 0.5106 - val_loss: 1.4076
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 29s 272ms/step - accuracy: 0.5086 - loss: 1.3551 - val_accuracy: 0.5248 - val_loss: 1.3167
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 24s 221ms/step - accuracy: 0.5505 - loss: 1.2383 - val_accuracy: 0.5225 - val_loss: 1.3206
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 29s 277ms/step - accuracy: 0.5774 - loss: 1.1780 - val_accuracy: 0.5366 - val_loss: 1.3162
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 24s 222ms/step - accuracy: 0.6176 - loss: 1.0859 - val_accuracy: 0.5189 - val_loss: 1.3481
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 24s 223ms/step - accuracy: 0.6265 - loss: 1.0202 - val_accuracy: 0.5366 - val_loss: 1.3330
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 23s 219ms/step - accuracy: 0.6678 - loss: 0.9473 - val_accuracy: 0.5686 - val_loss: 1.2790
Epoch 8/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 23s 209ms/step - accuracy: 0.6847 - loss: 0

In [40]:
loss,accuracy = model.evaluate(val_ds)
print("validation accuracy: " , accuracy)

27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 154ms/step - accuracy: 0.5686 - loss: 1.2790
validation accuracy:  0.5685579180717468


In [41]:
model.save("style_buddy_pattern_classifier-model4.keras")